In [1]:
# ============================================================
# CS781 Capstone - PHASE 5: Legal Statute Identification (LSI) - multi-label, Macro-F1
# Long document -> chunks -> one vector per chunk -> document pooling -> one sigmoid score per statute
# -> thresholds tuned on dev (global or per statute). Starts from YOUR Phase-1 model.
# Everything from scratch in plain PyTorch (no peft) - Phase 3+ rule.
#
# One notebook for all experiments - change Cell 1 only:
#   FT_METHOD: "full" | "partial" | "lora" | "qlora" | "lorma" | "adalora" | "adapter" | "prefix" | "prompt"
#   Knowledge distillation: set KD_TEACHER to a trained Phase-5 run's best_lsi.pt
# ============================================================

# %% 0. IMPORTS
import os, json, math, time, glob, random, zipfile
from collections import Counter
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PreTrainedTokenizerFast   # loads YOUR tokenizer file only

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

# %% 1. CONFIG  <-------------------------------------------------------- EDIT HERE
QUICK_TEST = False          # True = few docs, 1 epoch, to check the pipeline

MODEL_TYPE = "llama"        # "gpt" (Model 1) | "llama" (Model 2) | "gqa" (Model 3) | "hybrid" (Model 4)
PHASE1_CKPT = "/kaggle/input/datasets/kavyajain123987/phase1-models/best mdl2 phase1.pt"
TOKENIZER = "/kaggle/input/datasets/kavyajain123987/phase1/tokenizer (1).json"

FT_METHOD = "lora"          # full | partial | lora | qlora | lorma | adalora | adapter | prefix | prompt
FREEZE_BOTTOM = 4           # partial: freeze embeddings + this many lowest blocks
LORA_R, LORA_ALPHA = 8, 16  # lora / qlora / lorma
ADA_R_INIT, ADA_R_TARGET = 12, 4
ADAPTER_DIM = 64
N_PROMPT = 16               # prefix / prompt virtual tokens
QUANT_BLOCK = 64            # qlora 4-bit block size

# ---- long-document representation ----
SECTION = "spread"          # "spread" (chunks spread over the WHOLE document) | "head" | "tail" | "headtail"
MAX_LEN = 512
N_CHUNKS = 6
CHUNK_POOL = "last"         # "last" | "mean"
DOC_POOL = "attn"           # "attn" | "mean" | "max" (max over chunk vectors)

# ---- multi-label ----
LOSS = "bce_pos"            # "bce" | "bce_pos" (positive-weighted BCE, helps rare statutes) | "focal"
MAX_POS_WEIGHT = 20.0
THRESHOLDS = "per_label"    # "per_label" (tuned per statute on dev) | "global" (one tuned threshold)
MIN_DEV_POS = 5             # per-label threshold only for statutes with >= this many dev positives
AT_LEAST_ONE = True         # every document gets at least its top-scoring statute

# ---- training ----
MAX_TRAIN_DOCS = 10000
EPOCHS = 2
LR_BODY = 5e-5
LR_NEW = 1e-3
WEIGHT_DECAY = 0.01
DOCS_PER_STEP = 8
WIN_BATCH = 6
EVAL_DEV_DOCS = 1500        # dev subset during training; full dev at the end (thresholds)
SEED = 42

# ---- knowledge distillation (mandatory experiment) ----
KD_TEACHER = None           # path to a trained Phase-5 run's best_lsi.pt (file, .zip, or extracted folder)
KD_KEEP_EVERY = 2
KD_T, KD_ALPHA = 2.0, 0.5   # loss = ALPHA * task loss + (1-ALPHA) * BCE(student/T, teacher/T)

OUTPUT_DIR = f"/kaggle/working/phase5_{MODEL_TYPE}_{FT_METHOD}" + ("_kd" if KD_TEACHER else "")
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# %% 2. DOWNLOAD PHASE-5 DATA + LOOK AT IT
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["Phase_5_LSI/*"], local_dir="/tmp/cs781_data")
P5 = os.path.join(DATA_DIR, "Phase_5_LSI")
for root, _, files in os.walk(P5):
    for f in sorted(files):
        print(os.path.join(root, f), f"{os.path.getsize(os.path.join(root, f)) / 1e6:.1f} MB")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def peek(obj, n=300):
    s = json.dumps(obj, ensure_ascii=False)
    return s[:n] + ("..." if len(s) > n else "")

def parse(obj):
    """-> list of {"id", "text", "labels" (list of int or None)}"""
    if isinstance(obj, dict) and "data" in obj and isinstance(obj["data"], (list, dict)):
        obj = obj["data"]
    items = obj.items() if isinstance(obj, dict) else enumerate(obj)
    out = []
    for k, d in items:
        if isinstance(d, str):
            d = {"text": d}
        if not isinstance(d, dict):
            continue
        text = next((d[x] for x in ("text", "document", "case_text", "content") if x in d), None)
        if isinstance(text, list):
            text = " ".join(map(str, text))
        labs = next((d[x] for x in ("labels", "label", "statutes", "pred") if x in d), None)
        if labs is not None and not isinstance(labs, list):
            labs = [labs]
        out.append({"id": str(d.get("id", d.get("case_id", d.get("doc_id", k)))), "text": text or "",
                    "labels": sorted(set(int(x) for x in labs)) if labs is not None else None})
    return out

splits = {}
for name, path in [("train", "Train/train.json"), ("dev", "Train/dev.json"), ("test", "Test/test.json")]:
    raw = load_json(os.path.join(P5, path))
    print(f"\n{path}: {type(raw).__name__} with {len(raw)} entries | first: "
          f"{peek(raw[0] if isinstance(raw, list) else next(iter(raw.items())))}")
    splits[name] = parse(raw)
assert splits["test"] and all(d["text"] for d in splits["test"]), "test docs not parsed - check the format above"

LABELS = sorted({l for d in splits["train"] + splits["dev"] for l in d["labels"]})
LAB2IDX = {l: i for i, l in enumerate(LABELS)}
NL = len(LABELS)
lab_cnt = Counter(l for d in splits["train"] for l in d["labels"])
for name, ds in splits.items():
    chars = [len(d["text"]) for d in ds]
    nlab = [len(d["labels"]) for d in ds if d["labels"] is not None]
    print(f"{name}: {len(ds)} docs | chars median {int(np.median(chars))}, max {max(chars)}"
          + (f" | labels per doc mean {np.mean(nlab):.2f}, max {max(nlab)}" if nlab else ""))
print(f"{NL} statute IDs (range {min(LABELS)}..{max(LABELS)}) | train frequency: most common "
      f"{lab_cnt.most_common(3)}, rarest {sorted(lab_cnt.items(), key=lambda x: x[1])[:3]}")
dev_only = [l for l in LABELS if l not in lab_cnt]
if dev_only: print("statutes seen in dev but not in train:", dev_only)

random.Random(SEED).shuffle(splits["train"])
if MAX_TRAIN_DOCS:
    splits["train"] = splits["train"][:MAX_TRAIN_DOCS]
if QUICK_TEST:
    splits["train"], splits["dev"] = splits["train"][:40], splits["dev"][:30]
    EPOCHS = 1
print(f"using {len(splits['train'])} training docs, {len(splits['dev'])} dev docs")

# %% 3. TOKENIZE ONLY THE PART OF EACH DOCUMENT THAT IS READ
tokenizer = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER)
tokenizer.model_max_length = int(1e9)
BUDGET = N_CHUNKS * MAX_LEN
CHAR_BUDGET = BUDGET * 8

def select_ids(text):
    if SECTION == "tail":
        ids = tokenizer.encode(text[-CHAR_BUDGET:], add_special_tokens=False)[-BUDGET:]
    elif SECTION == "head":
        ids = tokenizer.encode(text[:CHAR_BUDGET], add_special_tokens=False)[:BUDGET]
    elif SECTION == "headtail":
        if len(text) <= CHAR_BUDGET:
            ids = tokenizer.encode(text, add_special_tokens=False)[-BUDGET:]
        else:
            h = tokenizer.encode(text[:CHAR_BUDGET // 2], add_special_tokens=False)[:BUDGET // 2]
            ids = h + tokenizer.encode(text[-CHAR_BUDGET // 2:], add_special_tokens=False)[-(BUDGET - len(h)):]
    elif SECTION == "spread":
        ids_all = tokenizer.encode(text, add_special_tokens=False)
        if len(ids_all) > BUDGET:
            starts = np.linspace(0, len(ids_all) - MAX_LEN, N_CHUNKS).astype(int)
            return [ids_all[s:s + MAX_LEN] for s in starts]
        ids = ids_all
    else:
        raise ValueError(SECTION)
    ids = ids or [0]
    return [ids[i:i + MAX_LEN] for i in range(0, len(ids), MAX_LEN)]

t0 = time.time()
for name, ds in splits.items():
    for d in ds:
        d["chunks"] = select_ids(d["text"])
        d["text"] = None
        if d["labels"] is not None:
            y = torch.zeros(NL)
            y[[LAB2IDX[l] for l in d["labels"] if l in LAB2IDX]] = 1.0
            d["y"] = y
    print(f"{name}: tokenized in {time.time() - t0:.0f}s | chunks per doc mean {np.mean([len(d['chunks']) for d in ds]):.1f}")

# %% 4. PHASE-1 MODEL DEFINITIONS (unchanged from Phase 1)
def build_model1(cfg):
    # ---------- Model 1: GPT-style ----------
    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            assert d_model % n_head == 0
            self.d_model = d_model
            self.n_head = n_head
            self.head_dim = d_model // n_head
            self.c_attn = nn.Linear(d_model, 3 * d_model)
            self.c_proj = nn.Linear(d_model, d_model)
            self.attn_dropout = dropout
            self.resid_dropout = nn.Dropout(dropout)

        def forward(self, x):
            B, T, C = x.shape
            qkv = self.c_attn(x)
            q, k, v = qkv.split(self.d_model, dim=2)
            q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=(self.attn_dropout if self.training else 0.0),
                is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            y = self.c_proj(y)
            y = self.resid_dropout(y)
            return y


    class MLP(nn.Module):
        def __init__(self, d_model, dropout):
            super().__init__()
            self.fc = nn.Linear(d_model, 4 * d_model)
            self.proj = nn.Linear(4 * d_model, d_model)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            x = self.fc(x)
            x = F.gelu(x)
            x = self.proj(x)
            x = self.dropout(x)
            return x


    class TransformerBlock(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            self.ln_1 = nn.LayerNorm(d_model)
            self.attn = CausalSelfAttention(d_model, n_head, dropout)
            self.ln_2 = nn.LayerNorm(d_model)
            self.mlp = MLP(d_model, dropout)

        def forward(self, x):
            x = x + self.attn(self.ln_1(x))
            x = x + self.mlp(self.ln_2(x))
            return x


    class LegalGPT(nn.Module):
        def __init__(self, vocab_size, block_size, n_layer, n_head, d_model, dropout):
            super().__init__()
            self.block_size = block_size
            self.wte = nn.Embedding(vocab_size, d_model)
            self.wpe = nn.Embedding(block_size, d_model)
            self.blocks = nn.ModuleList([TransformerBlock(d_model, n_head, dropout) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.wte.weight
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            B, T = idx.shape
            assert T <= self.block_size
            positions = torch.arange(0, T, device=idx.device)
            x = self.wte(idx) + self.wpe(positions)
            for block in self.blocks:
                x = block(x)
            x = self.ln_f(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return LegalGPT(**cfg)


def build_model2(cfg):
    # ---------- Model 2: LLaMA-style ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = CausalSelfAttention(d_model, n_heads, max_seq_len)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([LLaMABlock(cfg.d_model, cfg.n_heads, cfg.ffn_dim, cfg.max_seq_len)
                                         for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
        # (your generate() method is not needed: the pipeline's batched generate_batch() is used)
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model3(cfg):
    # ---------- Model 3: LLaMA + GQA + sliding window ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        half = x.shape[-1] // 2
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat([-x2, x1], dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class GQASlidingWindowAttention(nn.Module):
        """
        GQA: n_heads query heads share n_kv_heads key/value heads (8 Q -> 2 KV: each KV head serves 4 Q heads).
        Sliding window: token i attends only to tokens j with 0 <= i - j < window_size (causal + local).
        """
        def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, window_size):
            super().__init__()
            assert d_model % n_heads == 0
            assert n_heads % n_kv_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.n_kv_heads = n_kv_heads
            self.head_dim = d_model // n_heads
            self.window_size = window_size
            self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
            self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
            # mask[q, k] = True -> q may attend to k   (k <= q  AND  q - k < window)
            positions = torch.arange(max_seq_len)
            distance = positions[:, None] - positions[None, :]
            self.register_buffer("window_mask", (distance >= 0) & (distance < window_size), persistent=False)

        def forward(self, x):
            B, T, C = x.shape
            q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)     # [B, 8, T, D]
            k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            q, k = self.rope(q, k)
            repeat_factor = self.n_heads // self.n_kv_heads
            k = k.repeat_interleave(repeat_factor, dim=1)                                  # [B, 8, T, D]
            v = v.repeat_interleave(repeat_factor, dim=1)
            mask = self.window_mask[:T, :T]
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, max_seq_len, window_size):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = GQASlidingWindowAttention(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads,
                                                       max_seq_len=max_seq_len, window_size=window_size)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([
                LLaMABlock(d_model=cfg.d_model, n_heads=cfg.n_heads, n_kv_heads=cfg.n_kv_heads, ffn_dim=cfg.ffn_dim,
                           max_seq_len=cfg.max_seq_len, window_size=cfg.window_size)
                for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model4(cfg):
    # ---------- Model 4: hybrid attention + gated conv ----------
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles."""
        def __init__(self, dim, max_seq_len=2048, base=10000):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            self.base = base
            inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.shape[-2]
            if seq_len > self.cos_cached.shape[-2]:
                self._build_cache(seq_len)
            # FIX: cast to q's dtype. Otherwise fp16 q * fp32 cos -> fp32 q/k while v stays fp16,
            # and scaled_dot_product_attention can fail with a dtype mismatch.
            cos = self.cos_cached[:, :, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :, :seq_len, :].to(q.dtype)
            q = q * cos + rotate_half(q) * sin
            k = k * cos + rotate_half(k) * sin
            return q, k


    class RMSNorm(nn.Module):
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return x * rms * self.weight


    class SwiGLU(nn.Module):
        def __init__(self, d_model, hidden_dim):
            super().__init__()
            self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
            self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
            self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

        def forward(self, x):
            return self.w2(F.silu(self.w1(x)) * self.w3(x))


    class CausalSelfAttention(nn.Module):
        """GLOBAL mixer: token i attends to tokens 0..i."""
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len=max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class GatedCausalConv(nn.Module):
        """
        LOCAL mixer: Linear(D->2D) -> (value, gate);
        value -> causal depthwise Conv1d (left padding only: position t sees t-k+1 .. t, never t+1);
        out = Linear( conv(value) * sigmoid(gate) ).
        """
        def __init__(self, d_model, kernel_size=5):
            super().__init__()
            self.d_model = d_model
            self.kernel_size = kernel_size
            self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
            self.conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size, groups=d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)

        def forward(self, x):
            value, gate = self.in_proj(x).chunk(2, dim=-1)
            value = value.transpose(1, 2)                          # [B, D, T] for Conv1d
            value = F.pad(value, (self.kernel_size - 1, 0))        # LEFT padding only -> causal
            value = self.conv(value).transpose(1, 2)               # back to [B, T, D]
            return self.out_proj(value * torch.sigmoid(gate))


    class HybridBlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len, conv_kernel, use_attention):
            super().__init__()
            self.use_attention = use_attention
            self.norm1 = RMSNorm(d_model)
            self.mixer = (CausalSelfAttention(d_model=d_model, n_heads=n_heads, max_seq_len=max_seq_len)
                          if use_attention else GatedCausalConv(d_model=d_model, kernel_size=conv_kernel))
            self.norm2 = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model=d_model, hidden_dim=ffn_dim)

        def forward(self, x):
            x = x + self.mixer(self.norm1(x))
            x = x + self.ffn(self.norm2(x))
            return x


    class HybridLM(nn.Module):
        def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, max_seq_len, conv_kernel):
            super().__init__()
            self.vocab_size = vocab_size
            self.d_model = d_model
            self.token_embedding = nn.Embedding(vocab_size, d_model)
            # even layers -> attention, odd layers -> gated conv
            self.blocks = nn.ModuleList([
                HybridBlock(d_model=d_model, n_heads=n_heads, ffn_dim=ffn_dim, max_seq_len=max_seq_len,
                            conv_kernel=conv_kernel, use_attention=(i % 2 == 0))
                for i in range(n_layers)])
            self.final_norm = RMSNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, input_ids, targets=None):
            x = self.token_embedding(input_ids)
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                # FIX: the pipeline's targets are ALREADY shifted by one (y = x[1:]).
                # Your version shifted again (logits[:, :-1] vs targets[:, 1:]) -> the model
                # would learn to predict 2 tokens ahead. Same loss as Models 1-3 now.
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return HybridLM(**cfg)


BUILDERS = {"gpt": build_model1, "llama": build_model2, "gqa": build_model3, "hybrid": build_model4}

# %% 5. ADAPTATION METHODS (from scratch)
class LoRALinear(nn.Module):
    """LoRA:   y = W x + s * B A x,  s = alpha/r.   rsLoRA: same but s = alpha/sqrt(r) (rank-stabilised)."""
    def __init__(self, base, r, alpha, rank_stabilized=False):
        super().__init__()
        self.base = base
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative):  y = (I + s * B A) W x.  Check the exact variant against the LoRMA paper."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)


class AdaLoRALinear(nn.Module):
    """AdaLoRA:  y = W x + s * P diag(lambda * mask) Q x  (SVD-form update).
    Ranks are pruned during training by importance |lambda * dL/dlambda| (EMA), down to a global budget."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("mask", torch.ones(r))
        self.register_buffer("importance", torch.zeros(r))

    def forward(self, x):
        h = F.linear(x, self.Q) * (self.lam * self.mask)
        return self.base(x) + self.scale * F.linear(h, self.P)

    def orth_reg(self):
        r = self.lam.numel()
        I = torch.eye(r, device=self.P.device)
        return ((self.P.T @ self.P - I) ** 2).sum() + ((self.Q @ self.Q.T - I) ** 2).sum()


def wrap_linears(module, fn):
    """replace every nn.Linear `child` (attribute name `name`) by fn(child, name) unless it returns None"""
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            new = fn(child, name)
            if new is not None:
                setattr(module, name, new)
        else:
            wrap_linears(child, fn)

class Quant4Linear(nn.Module):
    """Frozen weight stored in 4 bits (QLoRA-style): blockwise absmax, 2 codes packed per byte.
    (QLoRA uses the NF4 code book; this uses uniform 4-bit levels -7..7 - say so in the report.)"""
    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.data.float()
        self.out_features, self.in_features = W.shape
        flat = W.flatten()
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        scale = flat.abs().amax(1, keepdim=True).clamp(min=1e-8) / 7.0
        q = torch.clamp(torch.round(flat / scale), -7, 7).to(torch.int16) + 8      # 1..15
        q = q.flatten().to(torch.uint8)
        if q.numel() % 2: q = torch.cat([q, q.new_zeros(1)])
        self.register_buffer("packed", (q[0::2] | (q[1::2] << 4)).contiguous())
        self.register_buffer("scale", scale.half())
        self.numel, self.block = W.numel(), block
        self.bias = None if base.bias is None else nn.Parameter(base.bias.data.clone(), requires_grad=False)

    def weight(self):
        q = torch.stack([self.packed & 15, self.packed >> 4], 1).flatten()[: self.scale.numel() * self.block]
        W = (q.view(-1, self.block).float() - 8.0) * self.scale.float()
        return W.flatten()[: self.numel].view(self.out_features, self.in_features)

    def forward(self, x):
        return F.linear(x, self.weight().to(x.dtype), None if self.bias is None else self.bias.to(x.dtype))

class AdapterBlock(nn.Module):
    """Houlsby-style bottleneck adapter after a frozen block: x + up(GELU(down(x)))"""
    def __init__(self, block, d_model, dim):
        super().__init__()
        self.block = block
        self.down, self.up = nn.Linear(d_model, dim), nn.Linear(dim, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        x = self.block(x)
        return x + self.up(F.gelu(self.down(x)))

# %% 6. CHUNK ENCODER (+ optional soft prompt / prefix) + DOCUMENT POOLING + MULTI-LABEL HEAD
def emb_module(bb):
    return bb.wte if hasattr(bb, "wte") else bb.token_embedding

def final_norm(bb):
    return bb.ln_f if hasattr(bb, "ln_f") else bb.final_norm


class LSIModel(nn.Module):
    def __init__(self, backbone, d_model, n_labels, doc_pool, method, n_prompt):
        super().__init__()
        self.bb, self.doc_pool, self.method = backbone, doc_pool, method
        self.n_prompt = n_prompt if method in ("prompt", "prefix") else 0
        if method == "prompt":
            self.prompt = nn.Parameter(torch.randn(n_prompt, d_model) * 0.02)
        if method == "prefix":
            self.prefix = nn.Parameter(torch.randn(len(backbone.blocks) + 1, n_prompt, d_model) * 0.02)
        if doc_pool == "attn":
            self.att = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))
        self.drop = nn.Dropout(0.1)
        self.cls = nn.Linear(d_model, n_labels)

    def hidden(self, ids):
        bb, P = self.bb, self.n_prompt
        x = emb_module(bb)(ids)
        if P:
            pre = self.prompt if self.method == "prompt" else self.prefix[0]
            x = torch.cat([pre.unsqueeze(0).expand(ids.size(0), -1, -1).to(x.dtype), x], 1)
        if hasattr(bb, "wpe"):
            x = x + bb.wpe(torch.arange(x.size(1), device=ids.device))
        for i, blk in enumerate(bb.blocks):
            if self.method == "prefix" and i > 0:
                x = torch.cat([self.prefix[i].unsqueeze(0).expand(ids.size(0), -1, -1).to(x.dtype), x[:, P:]], 1)
            x = blk(x)
        return final_norm(bb)(x)[:, P:]

    def forward(self, chunks, pool, autocast):
        vecs = []
        for b in range(0, len(chunks), WIN_BATCH):
            part = chunks[b:b + WIN_BATCH]
            T = max(map(len, part))
            x = torch.zeros(len(part), T, dtype=torch.long)
            for i, c in enumerate(part):
                x[i, :len(c)] = torch.tensor(c)
            with autocast():
                h = self.hidden(x.to(DEVICE))
            for i, c in enumerate(part):
                vecs.append(h[i, len(c) - 1] if pool == "last" else h[i, :len(c)].mean(0))
        v = torch.stack(vecs).float()
        if self.doc_pool == "attn":
            doc = (torch.softmax(self.att(v).squeeze(-1), 0).unsqueeze(-1) * v).sum(0)
        elif self.doc_pool == "max":
            doc = v.max(0).values
        else:
            doc = v.mean(0)
        return self.cls(self.drop(doc))            # (n_labels,) logits


def build_lsi_model(model_type, p1_cfg, p1_state, method, doc_pool, n_labels, keep_every=None):
    bb = BUILDERS[model_type](p1_cfg)
    if p1_state is not None:
        bb.load_state_dict(p1_state)
    bb.lm_head = nn.Identity()
    if keep_every:
        bb.blocks = nn.ModuleList(list(bb.blocks)[::keep_every])
    d = emb_module(bb).weight.shape[1]
    m = LSIModel(bb, d, n_labels, doc_pool, method, N_PROMPT)
    if method == "full":
        return m
    for p in bb.parameters(): p.requires_grad = False
    if method == "partial":
        for blk in bb.blocks[FREEZE_BOTTOM:]:
            for p in blk.parameters(): p.requires_grad = True
        for n, p in bb.named_parameters():
            if not n.startswith("blocks.") and p.dim() == 1: p.requires_grad = True
    elif method == "lora":
        wrap_linears(bb.blocks, lambda lin, n: LoRALinear(lin, LORA_R, LORA_ALPHA))
    elif method == "qlora":
        wrap_linears(bb.blocks, lambda lin, n: LoRALinear(Quant4Linear(lin, QUANT_BLOCK), LORA_R, LORA_ALPHA))
    elif method == "lorma":
        wrap_linears(bb.blocks, lambda lin, n: LoRMALinear(lin, LORA_R, LORA_ALPHA))
    elif method == "adalora":
        wrap_linears(bb.blocks, lambda lin, n: AdaLoRALinear(lin, ADA_R_INIT, LORA_ALPHA))
    elif method == "adapter":
        bb.blocks = nn.ModuleList([AdapterBlock(b, d, ADAPTER_DIM) for b in bb.blocks])
    elif method not in ("prefix", "prompt"):
        raise ValueError(method)
    return m

# %% 7. LOAD PHASE-1 WEIGHTS (+ KD teacher)
def as_ckpt_file(path):
    """Kaggle sometimes auto-extracts an uploaded .pt (it is a zip inside) into a FOLDER.
    If `path` is such a folder, re-pack it into a normal .pt file that torch.load can read."""
    if not os.path.isdir(path):
        return path
    root = next((r for r, _, f in os.walk(path) if "data.pkl" in f), None)
    assert root, f"{path} is a folder without data.pkl - not an extracted checkpoint"
    out = os.path.join("/kaggle/working", os.path.basename(path.rstrip("/")) + "_repacked.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for r, _, files in os.walk(root):
            for f in files:
                full = os.path.join(r, f)
                z.write(full, "archive/" + os.path.relpath(full, root))
    print(f"Re-packed extracted checkpoint folder {path} -> {out}")
    return out
ck = torch.load(as_ckpt_file(PHASE1_CKPT), map_location="cpu")
P1_CFG = ck["config"]
print("Phase-1 config:", P1_CFG)
assert P1_CFG["vocab_size"] == len(tokenizer), "Tokenizer doesn't match this Phase-1 model"
assert MAX_LEN + (N_PROMPT if FT_METHOD in ("prompt", "prefix") else 0) <= \
    P1_CFG.get("block_size", P1_CFG.get("max_seq_len")), "MAX_LEN (+prompt) > Phase-1 context"

USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
autocast = lambda: torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)

RUN_CFG = dict(model_type=MODEL_TYPE, ft_method=FT_METHOD, doc_pool=DOC_POOL, chunk_pool=CHUNK_POOL,
               section=SECTION, n_chunks=N_CHUNKS, max_len=MAX_LEN, labels=LABELS, lora_r=LORA_R,
               lora_alpha=LORA_ALPHA, ada_r_init=ADA_R_INIT, adapter_dim=ADAPTER_DIM, n_prompt=N_PROMPT,
               freeze_bottom=FREEZE_BOTTOM, keep_every=KD_KEEP_EVERY if KD_TEACHER else None, phase1_cfg=P1_CFG)

teacher = None
if KD_TEACHER:
    t = torch.load(as_ckpt_file(KD_TEACHER), map_location="cpu")
    tc = t["run_cfg"]
    assert (tc["section"], tc["n_chunks"], tc["max_len"]) == (SECTION, N_CHUNKS, MAX_LEN), \
        "teacher must read the same part of the document (SECTION / N_CHUNKS / MAX_LEN)"
    assert tc["labels"] == LABELS, "teacher was trained on a different statute label set"
    for k in ("lora_r", "lora_alpha", "ada_r_init", "adapter_dim", "n_prompt", "freeze_bottom"):
        globals()[k.upper()] = tc[k]
    teacher = build_lsi_model(tc["model_type"], tc["phase1_cfg"], None, tc["ft_method"], tc["doc_pool"], NL,
                              tc.get("keep_every"))
    teacher.load_state_dict(t["state"])
    teacher = teacher.to(DEVICE).eval()
    for p in teacher.parameters(): p.requires_grad = False
    T_POOL = tc["chunk_pool"]
    n_teacher = sum(p.numel() for p in teacher.parameters())
    print(f"KD teacher: {tc['model_type']} + {tc['ft_method']} ({n_teacher:,} params)")
    FT_METHOD = RUN_CFG["ft_method"] = "full"

model = build_lsi_model(MODEL_TYPE, P1_CFG, ck["model"], FT_METHOD, DOC_POOL, NL,
                        KD_KEEP_EVERY if KD_TEACHER else None).to(DEVICE)
del ck
n_total = sum(p.numel() for p in model.parameters()) + sum(q.numel for q in model.modules() if isinstance(q, Quant4Linear))
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{FT_METHOD}{' (KD student)' if KD_TEACHER else ''}: trainable {n_train:,} / {n_total:,} "
      f"({100 * n_train / n_total:.2f}%) | section={SECTION} {N_CHUNKS}x{MAX_LEN} | doc_pool={DOC_POOL} | loss={LOSS}")
if KD_TEACHER:
    print(f"KD: student {n_total:,} params = {100 * n_total / n_teacher:.0f}% of teacher")

# %% 8. METRICS + THRESHOLDS
def f1_scores(Y, P):
    """Y, P: (N, L) 0/1 numpy -> macro-F1 (labels present in Y or P), micro-F1, subset accuracy"""
    tp = (Y * P).sum(0); fp = ((1 - Y) * P).sum(0); fn = (Y * (1 - P)).sum(0)
    present = (Y.sum(0) + P.sum(0)) > 0
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    micro = 2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)
    return {"macro_f1": float(f1[present].mean()) if present.any() else 0.0, "micro_f1": float(micro),
            "subset_acc": float((Y == P).all(1).mean()), "per_label_f1": f1}

def decide(S, thr):
    P = (S >= thr).astype(np.float32)
    if AT_LEAST_ONE:
        empty = P.sum(1) == 0
        P[empty, S[empty].argmax(1)] = 1.0
    return P

GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)

def tune_thresholds(Y, S):
    """global threshold maximising macro-F1; then (optionally) per statute, maximising that statute's F1"""
    g = max(GRID, key=lambda t: (round(f1_scores(Y, decide(S, t))["macro_f1"], 6), -abs(t - 0.5)))
    thr = np.full(Y.shape[1], g, dtype=np.float32)
    if THRESHOLDS == "per_label":
        for j in range(Y.shape[1]):
            if Y[:, j].sum() < MIN_DEV_POS:
                continue
            def f1j(t):
                p = (S[:, j] >= t); tp = (p * Y[:, j]).sum()
                return 2 * tp / max(p.sum() + Y[:, j].sum(), 1)
            thr[j] = max(GRID, key=lambda t: (round(f1j(t), 6), -abs(t - g)))
    return g, thr

# %% 9. TRAINING
sub_cnt = Counter(l for d in splits["train"] for l in d["labels"])      # counts in the training subset used
pos = torch.tensor([sub_cnt.get(l, 0) for l in LABELS], dtype=torch.float32)
n_tr = len(splits["train"])
pos_weight = ((n_tr - pos) / pos.clamp(min=1)).clamp(1.0, MAX_POS_WEIGHT).to(DEVICE)   # (#neg / #pos), capped

def task_loss(logits, y):
    if LOSS == "bce":
        return F.binary_cross_entropy_with_logits(logits, y)
    if LOSS == "bce_pos":
        return F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
    if LOSS == "focal":                           # down-weights easy negatives (most statutes, most docs)
        p = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
        pt = p * y + (1 - p) * (1 - y)
        alpha = 0.75 * y + 0.25 * (1 - y)
        return (alpha * (1 - pt) ** 2 * ce).sum() / max(y.sum().item(), 1.0)
    raise ValueError(LOSS)

new_ids = {id(p) for n, p in model.named_parameters()
           if not n.startswith("bb.") or n.endswith((".A", ".B", ".P", ".Q", ".lam")) or ".down." in n or ".up." in n}
groups = [{"params": [p for p in model.parameters() if p.requires_grad and id(p) in new_ids], "lr": LR_NEW},
          {"params": [p for p in model.parameters() if p.requires_grad and id(p) not in new_ids], "lr": LR_BODY}]
optimizer = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY)
train_docs = splits["train"]
total_steps = EPOCHS * math.ceil(len(train_docs) / DOCS_PER_STEP)
warmup = max(1, int(0.06 * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, (total_steps - s) / max(1, total_steps - warmup)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)

ada_mods = [m_ for m_ in model.modules() if isinstance(m_, AdaLoRALinear)]
ADA_BUDGET0 = sum(m_.lam.numel() for m_ in ada_mods)
ADA_TARGET = ADA_R_TARGET * len(ada_mods)
ADA_T0, ADA_T1, ADA_EVERY, ADA_BETA = int(0.1 * total_steps), int(0.7 * total_steps), 10, 0.85
def adalora_update(step):
    for m_ in ada_mods:
        if m_.lam.grad is not None:
            m_.importance.mul_(ADA_BETA).add_((1 - ADA_BETA) * (m_.lam * m_.lam.grad).abs().detach())
    if step < ADA_T0 or step % ADA_EVERY: return
    prog = min(1.0, (step - ADA_T0) / max(1, ADA_T1 - ADA_T0))
    budget = int(ADA_TARGET + (ADA_BUDGET0 - ADA_TARGET) * (1 - prog) ** 3)
    allimp = torch.cat([m_.importance for m_ in ada_mods])
    if budget < allimp.numel():
        thr = torch.topk(allimp, budget).values[-1]
        for m_ in ada_mods:
            m_.mask.copy_((m_.importance >= thr).float())

@torch.no_grad()
def predict_scores(m, docs, pool=CHUNK_POOL):
    m.eval()
    return np.stack([torch.sigmoid(m(d["chunks"], pool, autocast).float()).cpu().numpy() for d in docs])

DEV_SUB = random.Random(SEED + 1).sample(splits["dev"], min(EVAL_DEV_DOCS or len(splits["dev"]), len(splits["dev"])))
BEST_PATH = os.path.join(OUTPUT_DIR, "best_lsi.pt")
EVAL_EVERY = max(1, math.ceil(len(train_docs) / DOCS_PER_STEP / 2))
best_f1, history, step, t0 = -1.0, [], 0, time.time()
if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()

def evaluate_and_maybe_save(tag):
    global best_f1
    Y = np.stack([d["y"].numpy() for d in DEV_SUB]); S = predict_scores(model, DEV_SUB)
    g, thr = tune_thresholds(Y, S)
    res = f1_scores(Y, decide(S, thr)); res.pop("per_label_f1")
    history.append({"at": tag, "global_thr": float(g), **res})
    print(f"  [{tag}] dev-subset Macro-F1 {res['macro_f1']:.4f} | Micro-F1 {res['micro_f1']:.4f} | "
          f"subset acc {res['subset_acc']:.4f} | global thr {g:.2f} | {time.time() - t0:.0f}s")
    if res["macro_f1"] > best_f1:
        best_f1 = res["macro_f1"]
        torch.save({"state": model.state_dict(), "run_cfg": RUN_CFG}, BEST_PATH)
        print("  >> best model saved")
    model.train()

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_docs)
    tot, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for i, d in enumerate(train_docs):
        logits = model(d["chunks"], CHUNK_POOL, autocast).float()
        y = d["y"].to(DEVICE)
        loss = task_loss(logits, y)
        if teacher is not None:
            with torch.no_grad():
                t_logits = teacher(d["chunks"], T_POOL, autocast).float()
            kd = F.binary_cross_entropy_with_logits(logits / KD_T, torch.sigmoid(t_logits / KD_T))
            loss = KD_ALPHA * loss + (1 - KD_ALPHA) * KD_T ** 2 * kd
        if ada_mods:
            loss = loss + 0.1 * sum(m_.orth_reg() for m_ in ada_mods) / len(ada_mods)
        scaler.scale(loss / DOCS_PER_STEP).backward()
        tot += loss.item(); n += 1
        if (i + 1) % DOCS_PER_STEP == 0 or i + 1 == len(train_docs):
            scaler.unscale_(optimizer)
            if ada_mods: adalora_update(step)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.grad is not None], 1.0)
            scaler.step(optimizer); scaler.update(); sched.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            if step % 50 == 0:
                rate = (time.time() - t0) / step
                print(f"epoch {epoch} step {step}/{total_steps} | loss {tot / n:.4f} | "
                      f"ETA training {rate * (total_steps - step) / 60:.0f} min")
            if step % EVAL_EVERY == 0:
                evaluate_and_maybe_save(f"epoch {epoch} step {step}")

TRAIN_MIN = (time.time() - t0) / 60
PEAK_GB = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else None
print(f"\nBest dev-subset Macro-F1 {best_f1:.4f} | {TRAIN_MIN:.1f} min | peak GPU {PEAK_GB} GB")

# full dev: final thresholds + analysis
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state"])
Yd = np.stack([d["y"].numpy() for d in splits["dev"]]); Sd = predict_scores(model, splits["dev"])
G_THR, THR = tune_thresholds(Yd, Sd)
res_g = f1_scores(Yd, decide(Sd, np.full(NL, G_THR)))
best_res = f1_scores(Yd, decide(Sd, THR))
per = best_res.pop("per_label_f1"); res_g.pop("per_label_f1")
torch.save({**ckpt, "thresholds": THR.tolist(), "global_threshold": float(G_THR)}, BEST_PATH)
print(f"FULL dev ({len(Yd)} docs): Macro-F1 {best_res['macro_f1']:.4f} ({THRESHOLDS} thresholds) | "
      f"with one global threshold {G_THR:.2f}: {res_g['macro_f1']:.4f} | Micro-F1 {best_res['micro_f1']:.4f} | "
      f"subset acc {best_res['subset_acc']:.4f}")
freq_order = sorted(range(NL), key=lambda j: -lab_cnt.get(LABELS[j], 0))
third = max(1, NL // 3)
for name, idx in [("frequent", freq_order[:third]), ("medium", freq_order[third:2 * third]), ("rare", freq_order[2 * third:])]:
    print(f"  {name} statutes ({len(idx)}): mean dev F1 {np.mean(per[idx]):.3f}")

# %% 10. TEST PREDICTIONS + prediction.json + ZIP
t1 = time.time()
St = predict_scores(model, splits["test"])
INFER_MS = (time.time() - t1) / len(splits["test"]) * 1000
Pt = decide(St, THR)
preds = [{"id": d["id"], "pred": sorted(int(LABELS[j]) for j in np.nonzero(row)[0])}
         for d, row in zip(splits["test"], Pt)]
assert len(preds) == len(splits["test"])
assert all(len(p["pred"]) == len(set(p["pred"])) and all(isinstance(x, int) for x in p["pred"]) for p in preds)
print(f"{len(preds)} test docs | statutes per doc mean {np.mean([len(p['pred']) for p in preds]):.2f} | "
      f"{INFER_MS:.0f} ms per doc | example {preds[0]}")

pred_path = os.path.join(OUTPUT_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f)
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")
print("Submission ready:", zip_path)

# %% 11. RUN SUMMARY (experiment log / report)
with open(os.path.join(OUTPUT_DIR, "run_summary.json"), "w") as f:
    json.dump({**{k: v for k, v in RUN_CFG.items() if k not in ("phase1_cfg", "labels")}, "kd_teacher": KD_TEACHER,
               "loss": LOSS, "thresholds": THRESHOLDS, "trainable_params": n_train, "total_params": n_total,
               "teacher_params": n_teacher if KD_TEACHER else None, "train_docs": len(train_docs),
               "epochs": EPOCHS, "dev_full": best_res, "dev_full_global_threshold": res_g,
               "global_threshold": float(G_THR), "history": history,
               "per_label_dev_f1": {int(LABELS[j]): round(float(per[j]), 4) for j in range(NL)},
               "train_minutes": round(TRAIN_MIN, 1), "peak_gpu_gb": PEAK_GB, "inference_ms_per_doc": round(INFER_MS, 1)},
              f, indent=2)
print("Done.")

Device: cuda Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

/tmp/cs781_data/Phase_5_LSI/Train/dev.json 79.4 MB
/tmp/cs781_data/Phase_5_LSI/Train/train.json 299.4 MB
/tmp/cs781_data/Phase_5_LSI/Test/test.json 98.1 MB

Train/train.json: list with 42750 entries | first: {"id": "1000008", "text": "(a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of industry, commerceand trade in and around the city of Mumbai which generatessubstantial quantity of wealth, there has been in...

Train/dev.json: list with 10181 entries | first: {"id": "1000139", "text": "Dr. <ENTITY>, J. The accused persons are the <ENTITY> andbrother respectively of the deceased. The incident purportedly took place on 3.1.1990.Though the prosecution relied on the evidence of many persons who supposedly witnessedthe occurrence, while deposing in Court, mos...

Test/test.json: list with 13019 entries | first: {"id": "100002997", "text": "2.Prosecution case in brief was that <ENTITY>, daughter